# EdgeVision: CNNs, MobileNetV2, and INT8 CPU Inference

This experiment trains two image classifiers, evaluates them on a held-out test split, then applies calibrated post-training static INT8 quantization and benchmarks CPU inference. It records accuracy, serialized model size, and latency so the trade-offs are visible side by side.

**Data choices:** CIFAR-10 downloads automatically, or select a custom `ImageFolder` dataset in the configuration cell. For custom data, use `train/<class-name>/*.jpg` and optionally `val/<class-name>/*.jpg` and `test/<class-name>/*.jpg`.

**Reproducibility:** seed, data splits, subset selection, and loader order are fixed by the configuration. PyTorch, torchvision, torchao, Python, and hardware details are recorded with the results. Deterministic kernels can be slower and some operations may still vary slightly by platform.


## 1. Environment and experiment settings

Install dependencies from `requirements.txt` in the same folder before starting this notebook. The first CIFAR-10 run and pretrained MobileNetV2 initialization may download data and weights. Defaults use a small, stratified sample and three epochs; raise the limits or set them to `None` for a fuller experiment. Training uses CUDA when available; all deployment benchmarks run on CPU.


In [1]:
from pathlib import Path
import os
import platform
import random
import sys
import time
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision
from torchvision import datasets, transforms
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2

SEED = 42
DATASET_MODE = "cifar10"  # "cifar10" or "custom"
CUSTOM_DATA_DIR = Path("data/your_images")
OUTPUT_DIR = Path("artifacts")

IMAGE_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 3e-4
NUM_WORKERS = 0  # portable default, especially on Windows/Jupyter
TRAIN_LIMIT = 5_000  # set None to use every training image
EVAL_LIMIT = 2_000   # set None to use every validation/test image
CALIBRATION_IMAGES = 256
BENCHMARK_WARMUP = 20
BENCHMARK_ITERATIONS = 100
BENCHMARK_BATCH_SIZE = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:  # older supported PyTorch versions
        torch.use_deterministic_algorithms(True)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python: {platform.python_version()} | PyTorch: {torch.__version__} | torchvision: {torchvision.__version__}")
print(f"Device for training: {DEVICE} | CPU threads: {torch.get_num_threads()}")


Python: 3.12.10 | PyTorch: 2.15.0+cpu | torchvision: 0.29.2+cpu
Device for training: cpu | CPU threads: 4


## 2. Load and prepare data

Both models use the same class mapping, image size, splits, normalization, and sampled examples. If custom validation and/or test folders are absent, the notebook creates deterministic class-stratified splits from the training folder. CIFAR-10 keeps its official test set and creates validation data from its training set.


In [2]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def stratified_indices(labels, fractions, seed=SEED):
    # Split indices per class; fractions must sum to 1.
    if not np.isclose(sum(fractions), 1.0):
        raise ValueError("Split fractions must sum to 1.")
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    buckets = [[] for _ in fractions]
    for label in np.unique(labels):
        indices = np.flatnonzero(labels == label)
        rng.shuffle(indices)
        counts = [int(round(len(indices) * fraction)) for fraction in fractions[:-1]]
        counts.append(len(indices) - sum(counts))
        start = 0
        for bucket, count in zip(buckets, counts):
            bucket.extend(indices[start:start + count].tolist())
            start += count
    for bucket in buckets:
        rng.shuffle(bucket)
    return buckets

def dataset_labels(dataset):
    if hasattr(dataset, "targets"):
        return np.asarray(dataset.targets)
    if isinstance(dataset, Subset):
        return dataset_labels(dataset.dataset)[np.asarray(dataset.indices)]
    raise TypeError(f"Cannot read labels from {type(dataset).__name__}")

def limited_subset(dataset, limit, seed):
    if limit is None or len(dataset) <= limit:
        return dataset
    labels = dataset_labels(dataset)
    fractions = [limit / len(dataset), 1.0 - limit / len(dataset)]
    selected, _ = stratified_indices(labels, fractions, seed)
    return Subset(dataset, selected)

def make_cifar10_loaders():
    train_aug = datasets.CIFAR10("data", train=True, download=True, transform=train_transform)
    train_eval = datasets.CIFAR10("data", train=True, download=True, transform=eval_transform)
    official_test = datasets.CIFAR10("data", train=False, download=True, transform=eval_transform)
    train_idx, val_idx = stratified_indices(train_aug.targets, [0.9, 0.1])
    train_ds = Subset(train_aug, train_idx)
    val_ds = Subset(train_eval, val_idx)
    class_names = train_aug.classes
    return train_ds, val_ds, official_test, class_names

def make_custom_loaders(root: Path):
    train_dir = root / "train"
    val_dir = root / "val"
    test_dir = root / "test"
    if not train_dir.is_dir():
        raise FileNotFoundError(f"Expected a training folder at {train_dir.resolve()}")

    train_aug = datasets.ImageFolder(train_dir, transform=train_transform)
    train_eval = datasets.ImageFolder(train_dir, transform=eval_transform)
    class_names = train_aug.classes
    if len(class_names) < 2:
        raise ValueError("Image classification requires at least two class folders.")

    val_present = val_dir.is_dir()
    test_present = test_dir.is_dir()
    if val_present:
        val_full = datasets.ImageFolder(val_dir, transform=eval_transform)
        if val_full.class_to_idx != train_aug.class_to_idx:
            raise ValueError("Class folders in val/ must match train/ exactly.")
    else:
        val_full = None
    if test_present:
        test_full = datasets.ImageFolder(test_dir, transform=eval_transform)
        if test_full.class_to_idx != train_aug.class_to_idx:
            raise ValueError("Class folders in test/ must match train/ exactly.")
    else:
        test_full = None

    if not val_present and not test_present:
        train_idx, val_idx, test_idx = stratified_indices(train_aug.targets, [0.8, 0.1, 0.1])
        train_ds = Subset(train_aug, train_idx)
        val_ds = Subset(train_eval, val_idx)
        test_ds = Subset(train_eval, test_idx)
    elif not val_present:
        train_idx, val_idx = stratified_indices(train_aug.targets, [0.9, 0.1])
        train_ds = Subset(train_aug, train_idx)
        val_ds = Subset(train_eval, val_idx)
        test_ds = test_full
    elif not test_present:
        train_idx, test_idx = stratified_indices(train_aug.targets, [0.9, 0.1])
        train_ds = Subset(train_aug, train_idx)
        val_ds = val_full
        test_ds = Subset(train_eval, test_idx)
    else:
        train_ds, val_ds, test_ds = train_aug, val_full, test_full
    return train_ds, val_ds, test_ds, class_names

if DATASET_MODE.lower() == "cifar10":
    train_dataset, val_dataset, test_dataset, class_names = make_cifar10_loaders()
elif DATASET_MODE.lower() == "custom":
    train_dataset, val_dataset, test_dataset, class_names = make_custom_loaders(CUSTOM_DATA_DIR)
else:
    raise ValueError("DATASET_MODE must be either 'cifar10' or 'custom'.")

if len(class_names) < 2:
    raise ValueError("The selected dataset must contain at least two classes.")

train_dataset = limited_subset(train_dataset, TRAIN_LIMIT, SEED + 1)
val_dataset = limited_subset(val_dataset, EVAL_LIMIT, SEED + 2)
test_dataset = limited_subset(test_dataset, EVAL_LIMIT, SEED + 3)

def make_loader(dataset, *, shuffle=False, batch_size=BATCH_SIZE, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"), generator=generator, drop_last=False,
    )

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset)
test_loader = make_loader(test_dataset)
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Images | train: {len(train_dataset):,} | validation: {len(val_dataset):,} | test: {len(test_dataset):,}")


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1010)>

In [ ]:
# Visualize a small batch after undoing ImageNet normalization.
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    image = image.permute(1, 2, 0).numpy() * np.asarray(STD) + np.asarray(MEAN)
    ax.imshow(np.clip(image, 0, 1))
    ax.set_title(class_names[int(label)])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Define the custom CNN and MobileNetV2

The custom network uses convolution, batch normalization, ReLU, pooling, and a compact classifier. MobileNetV2 starts from ImageNet weights and replaces its final classification layer. Both produce one logit per dataset class.


In [ ]:
class EdgeCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

def build_models(num_classes):
    custom = EdgeCNN(num_classes)
    mobile = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    mobile.classifier[1] = nn.Linear(mobile.last_channel, num_classes)
    return {"Custom CNN": custom, "MobileNetV2": mobile}

models = build_models(len(class_names))
for name, model in models.items():
    sample_logits = model(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE))
    assert sample_logits.shape == (2, len(class_names)), f"Unexpected output shape from {name}: {sample_logits.shape}"
    print(f"{name}: {sum(p.numel() for p in model.parameters()):,} parameters; output shape {tuple(sample_logits.shape)}")


## 4. Train and validate

Each epoch records training loss, training accuracy, validation loss, and validation accuracy. The checkpoint with the best validation accuracy is retained for the final test and quantization steps. MobileNetV2 is fine-tuned end to end.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train(training)
    loss_sum = 0.0
    correct = 0
    total = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                optimizer.step()
            loss_sum += float(loss.detach()) * y.size(0)
            correct += int((logits.argmax(1) == y).sum())
            total += y.size(0)
    if total == 0:
        raise ValueError("A data split is empty. Add more images or adjust the split/sample limits.")
    return {"loss": loss_sum / total, "accuracy": correct / total}

def train_model(model, name, train_loader, val_loader, epochs=EPOCHS):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    best_accuracy = -1.0
    best_state = copy.deepcopy(model.state_dict())
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_epoch(model, val_loader, criterion)
        history.append({"epoch": epoch, "train_loss": train_metrics["loss"],
                        "train_accuracy": train_metrics["accuracy"],
                        "val_loss": val_metrics["loss"], "val_accuracy": val_metrics["accuracy"]})
        if val_metrics["accuracy"] > best_accuracy:
            best_accuracy = val_metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())
        print(f"{name} | epoch {epoch:02d}/{epochs} | "
              f"train loss {train_metrics['loss']:.4f}, acc {train_metrics['accuracy']:.3f} | "
              f"val loss {val_metrics['loss']:.4f}, acc {val_metrics['accuracy']:.3f}")
    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(history)

trained_models = {}
histories = {}
for model_name, model in models.items():
    print(f"\nTraining {model_name}")
    trained_models[model_name], histories[model_name] = train_model(
        model, model_name, train_loader, val_loader
    )


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, history in histories.items():
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label=f"{name} train")
    axes[0].plot(history["epoch"], history["val_loss"], marker="o", linestyle="--", label=f"{name} val")
    axes[1].plot(history["epoch"], history["train_accuracy"], marker="o", label=f"{name} train")
    axes[1].plot(history["epoch"], history["val_accuracy"], marker="o", linestyle="--", label=f"{name} val")
axes[0].set(title="Loss by epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[1].set(title="Accuracy by epoch", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.show()


## 5. Float model test metrics

Evaluate each best-validation checkpoint on the held-out test split. These results establish the accuracy baseline before compression.


In [ ]:
float_test_metrics = {}
for name, model in trained_models.items():
    metrics = run_epoch(model, test_loader, nn.CrossEntropyLoss(), device=DEVICE)
    float_test_metrics[name] = metrics
    print(f"{name}: test accuracy={metrics['accuracy']:.4f}, test loss={metrics['loss']:.4f}")


## 6. Static INT8 post-training quantization for CPU

On PyTorch 2.6+ with torchao, this uses the PT2E x86 quantizer and lowers the result through Inductor. Older installations use PyTorch's built-in FX static quantization, which produces an eager CPU INT8 model. Both paths calibrate on representative training images and leave the test split untouched.

The FX path is used here if torchao is absent or PyTorch is older than 2.6. Quantization operators and speedups depend on the installed PyTorch build and CPU backend.


In [ ]:
def quantize_for_x86(model, calibration_loader, image_size=IMAGE_SIZE):
    model = copy.deepcopy(model).to("cpu").eval()
    example = torch.zeros(1, 3, image_size, image_size, dtype=torch.float32)
    seen = 0

    # Prefer the current PT2E flow when it exists; retain an FX path for older PyTorch.
    pt2e_available = tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2, 6)
    try:
        if not pt2e_available:
            raise ImportError("PT2E workflow requires PyTorch 2.6 or newer")
        from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
        from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import (
            X86InductorQuantizer,
            get_default_x86_inductor_quantization_config,
        )
        quantizer = X86InductorQuantizer()
        quantizer.set_global(get_default_x86_inductor_quantization_config())
        with torch.no_grad():
            exported = torch.export.export(model, (example,)).module()
        prepared = prepare_pt2e(exported, quantizer)
        mode = "PT2E INT8 + Inductor"
        uses_inductor = True
    except ImportError:
        from torch.ao.quantization import get_default_qconfig_mapping
        from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
        backend = "x86" if "x86" in torch.backends.quantized.supported_engines else "fbgemm"
        torch.backends.quantized.engine = backend
        prepared = prepare_fx(model, get_default_qconfig_mapping(backend), example_inputs=(example,))
        mode = f"FX static INT8 ({backend})"
        uses_inductor = False

    with torch.inference_mode():
        for x, _ in calibration_loader:
            batch = x[:max(0, CALIBRATION_IMAGES - seen)].to("cpu")
            if batch.shape[0] == 0:
                break
            for image in batch:
                prepared(image.unsqueeze(0).contiguous(memory_format=torch.channels_last))
                seen += 1
                if seen >= CALIBRATION_IMAGES:
                    break
            if seen >= CALIBRATION_IMAGES:
                break
    if seen == 0:
        raise ValueError("No calibration samples were available.")

    if uses_inductor:
        converted = convert_pt2e(prepared).eval()
        inference_model = torch.compile(converted, backend="inductor")
    else:
        converted = convert_fx(prepared).eval()
        inference_model = converted
    return converted, inference_model, seen, mode, uses_inductor

quantized_models = {}
calibration_counts = {}
quantization_modes = {}
uses_inductor_by_model = {}
for name, model in trained_models.items():
    converted, inference_model, seen, mode, uses_inductor = quantize_for_x86(model, train_loader)
    quantized_models[name] = {"converted": converted, "inference": inference_model}
    calibration_counts[name] = seen
    quantization_modes[name] = mode
    uses_inductor_by_model[name] = uses_inductor
    print(f"{name}: {mode}; calibrated on {seen} training images")


## 7. Compare accuracy, model size, and CPU latency

The benchmark uses batch size 1 for all models and the same normalized test examples. The PT2E path uses Inductor for both float and INT8 models; the compatibility FX path uses eager CPU inference for both. Warmup calls are excluded from timed iterations. Timing is reported per image in milliseconds; observed values vary by CPU, compiler, and thread count.

Model size is measured by serializing each model's state dict in the same PyTorch format. This compares the model weight payload; it is not a complete platform-specific deployment bundle.


In [ ]:
def serialized_state_dict_size_mb(model, path: Path):
    # Same serialization format for both precisions; FX quantized weights use packed INT8 tensors.
    torch.save(model.to("cpu").eval().state_dict(), path)
    return path.stat().st_size / (1024 ** 2)

def benchmark_latency(model, example, warmup=BENCHMARK_WARMUP, iterations=BENCHMARK_ITERATIONS):
    model.eval()
    with torch.inference_mode():
        for _ in range(warmup):
            model(example)
        timings = []
        for _ in range(iterations):
            start = time.perf_counter()
            model(example)
            timings.append(time.perf_counter() - start)
    return float(np.median(timings) * 1000.0 / example.shape[0])

def evaluate_single_image(model, loader):
    return run_epoch(model, loader, nn.CrossEntropyLoss(), device="cpu")

if BENCHMARK_BATCH_SIZE != 1:
    raise ValueError("Keep BENCHMARK_BATCH_SIZE=1 for the exported quantized model shape.")
cpu_test_loader = make_loader(test_dataset, batch_size=BENCHMARK_BATCH_SIZE)
example_batch, _ = next(iter(cpu_test_loader))
example_cpu = example_batch.contiguous(memory_format=torch.channels_last)
results = []

for name, float_model in trained_models.items():
    float_cpu = copy.deepcopy(float_model).to("cpu").eval()
    int8_model = quantized_models[name]["converted"].eval()
    if uses_inductor_by_model[name]:
        float_inference = torch.compile(float_cpu, backend="inductor")
        int8_inference = quantized_models[name]["inference"]
    else:
        float_inference = float_cpu
        int8_inference = quantized_models[name]["inference"]

    # Trigger any compilation before timing.
    with torch.inference_mode():
        float_inference(example_cpu)
        int8_inference(example_cpu)

    float_metrics = evaluate_single_image(float_inference, cpu_test_loader)
    int8_metrics = evaluate_single_image(int8_inference, cpu_test_loader)
    stem = name.lower().replace(" ", "_")
    float_path = OUTPUT_DIR / f"{stem}_float_state_dict.pth"
    int8_path = OUTPUT_DIR / f"{stem}_int8_state_dict.pth"
    float_size = serialized_state_dict_size_mb(float_cpu, float_path)
    int8_size = serialized_state_dict_size_mb(int8_model, int8_path)
    float_latency = benchmark_latency(float_inference, example_cpu)
    int8_latency = benchmark_latency(int8_inference, example_cpu)
    results.extend([
        {"model": name, "precision": "FP32", "quantization_path": "float baseline",
         "test_accuracy": float_metrics["accuracy"], "test_loss": float_metrics["loss"],
         "serialized_state_dict_mb": float_size, "cpu_latency_ms_per_image": float_latency},
        {"model": name, "precision": "INT8 PTQ", "quantization_path": quantization_modes[name],
         "test_accuracy": int8_metrics["accuracy"], "test_loss": int8_metrics["loss"],
         "serialized_state_dict_mb": int8_size, "cpu_latency_ms_per_image": int8_latency},
    ])

results_df = pd.DataFrame(results)
results_df["size_change_pct"] = results_df.groupby("model")["serialized_state_dict_mb"].transform(
    lambda values: (values / values.iloc[0] - 1) * 100
)
results_df["latency_change_pct"] = results_df.groupby("model")["cpu_latency_ms_per_image"].transform(
    lambda values: (values / values.iloc[0] - 1) * 100
)
display(results_df.round(4))
print(f"CPU threads: {torch.get_num_threads()} | benchmark batch size: {BENCHMARK_BATCH_SIZE}")


In [ ]:
# Accuracy, serialized artifact size, and CPU latency trade-offs.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
metrics_to_plot = [
    ("test_accuracy", "Test accuracy", "Accuracy"),
    ("serialized_state_dict_mb", "Serialized model size", "Size (MiB)"),
    ("cpu_latency_ms_per_image", "CPU latency (batch 1)", "Median ms / image"),
]
for ax, (column, title, ylabel) in zip(axes, metrics_to_plot):
    results_df.pivot(index="model", columns="precision", values=column).plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=15)
    ax.legend(title="Precision")
plt.tight_layout()
plt.show()


## 8. Interpret the results

Use the table and plots to answer:

1. Which model has the strongest held-out accuracy before quantization?
2. How much serialized size changes after INT8 quantization for each architecture?
3. Does INT8 reduce CPU latency on this machine, and does the result differ by architecture?
4. Is any accuracy loss acceptable for the measured size or latency benefit?

Quantization does not guarantee a speedup: CPU instruction support, backend coverage, graph compilation, and model size all affect the result. The short default training run is intended to make the workflow approachable; use a larger sample and more epochs before drawing conclusions about model quality.


## 9. Reproducibility record and artifacts

The following cell saves the comparison table and writes an environment/configuration record beside the exported models. Keep this record with results when comparing runs. For the same data, seed, configuration, software build, and hardware, rerunning the notebook should reproduce the experiment as closely as the platform's deterministic kernels allow.


In [ ]:
import importlib.metadata as metadata

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"

environment_record = {
    "seed": SEED,
    "dataset_mode": DATASET_MODE,
    "custom_data_dir": str(CUSTOM_DATA_DIR) if DATASET_MODE == "custom" else None,
    "classes": class_names,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_limit": TRAIN_LIMIT,
    "eval_limit": EVAL_LIMIT,
    "calibration_images": CALIBRATION_IMAGES,
    "benchmark_warmup": BENCHMARK_WARMUP,
    "benchmark_iterations": BENCHMARK_ITERATIONS,
    "benchmark_batch_size": BENCHMARK_BATCH_SIZE,
    "cpu_threads": torch.get_num_threads(),
    "training_device": str(DEVICE),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torchao": package_version("torchao"),
    "numpy": np.__version__,
}
results_df.to_csv(OUTPUT_DIR / "edgevision_results.csv", index=False)
pd.concat(histories, names=["model"]).reset_index().to_csv(OUTPUT_DIR / "edgevision_training_history.csv", index=False)
pd.Series(environment_record, name="value").to_json(OUTPUT_DIR / "edgevision_run_config.json", indent=2)
print(f"Saved artifacts to: {OUTPUT_DIR.resolve()}")
